In [4]:
import nltk
import random
import numpy as np
from nltk.corpus import cess_esp
corpus_sentences=list(cess_esp.tagged_sents())
number_sentences=len(corpus_sentences)

norm_corpus = []
for sentence in corpus_sentences:
    norm_sentence = []
    for w,cat in sentence:
        if w != '*0*':
            if str(cat).startswith('v'):
                cat = cat[0:3] if len(cat) >=3 else cat
            elif str(cat).startswith('F'):
                cat = cat[0:3] if len(cat) >=3 else cat
            else:
                cat = cat[0:2] if len(cat) >=3 else cat
            norm_sentence.append((w,cat))
    norm_corpus.append(norm_sentence)
print(norm_corpus[0:2])
            

[[('El', 'da'), ('grupo', 'nc'), ('estatal', 'aq'), ('Electricité_de_France', 'np'), ('-Fpa-', 'Fpa'), ('EDF', 'np'), ('-Fpt-', 'Fpt'), ('anunció', 'vmi'), ('hoy', 'rg'), (',', 'Fc'), ('jueves', 'W'), (',', 'Fc'), ('la', 'da'), ('compra', 'nc'), ('del', 'sp'), ('51_por_ciento', 'Zp'), ('de', 'sp'), ('la', 'da'), ('empresa', 'nc'), ('mexicana', 'aq'), ('Electricidad_Águila_de_Altamira', 'np'), ('-Fpa-', 'Fpa'), ('EAA', 'np'), ('-Fpt-', 'Fpt'), (',', 'Fc'), ('creada', 'aq'), ('por', 'sp'), ('el', 'da'), ('japonés', 'aq'), ('Mitsubishi_Corporation', 'np'), ('para', 'sp'), ('poner_en_marcha', 'vmn'), ('una', 'di'), ('central', 'nc'), ('de', 'sp'), ('gas', 'nc'), ('de', 'sp'), ('495', 'Z'), ('megavatios', 'nc'), ('.', 'Fp')], [('Una', 'di'), ('portavoz', 'nc'), ('de', 'sp'), ('EDF', 'np'), ('explicó', 'vmi'), ('a', 'sp'), ('EFE', 'np'), ('que', 'cs'), ('el', 'da'), ('proyecto', 'nc'), ('para', 'sp'), ('la', 'da'), ('construcción', 'nc'), ('de', 'sp'), ('Altamira_2', 'np'), (',', 'Fc'), ('al

In [5]:
#split corpus in test and train
import math

train = norm_corpus[0:math.floor(number_sentences*0.9)]
test = norm_corpus[math.ceil(number_sentences*0.9):]

print('train set size:', len(train))
print('test set size', len(test))

train set size: 5427
test set size 603


In [6]:
from nltk.tag import hmm
from nltk.tag import tnt

tagger_hmm=hmm.HiddenMarkovModelTagger.train(train)
print(tagger_hmm.accuracy(test))
tagger_tnt = tnt.TnT()
tagger_tnt.train(train)
print(tagger_tnt.accuracy(test))

0.8784427571832664
0.8255173440524044


In [7]:
def interval_trust(P, N):
    return 1.96*math.sqrt((P*(1-P))/N)

In [8]:
def tagger(train, test):
    local_hmm_tagger = hmm.HiddenMarkovModelTagger.train(train)
    hmm_tagger_accuracy = local_hmm_tagger.accuracy(test)
    local_tnt_tagger = tnt.TnT()
    local_tnt_tagger.train(train)
    tnt_tagger_accuracy = local_tnt_tagger.accuracy(test)
    return hmm_tagger_accuracy, tnt_tagger_accuracy

In [14]:
def fold_cross_validation(fold=10, shuffle=False):
    fold_cross_results = []
    local_corpus = norm_corpus.copy()
    if shuffle:
        random.shuffle(local_corpus)
    len_subsets = math.floor(number_sentences/fold)

    for i in range(fold):
        start = i*len_subsets
        end = (i+1)*len_subsets if i < fold-1 else -1
        _test = local_corpus[start:end]
        if(start != 0 and end != -1):
            _train = local_corpus[0:start] + local_corpus[end:]
        elif start == 0:
            _train = local_corpus[end:]
        elif end == -1:
            _train = local_corpus[0:start]
        prec_hmm, prec_tnt = tagger(_train, _test)
        fold_cross_results.append(np.array([prec_hmm, prec_tnt]))
    return np.array(fold_cross_results)
        


In [15]:
fold_cross_shuffle = fold_cross_validation(10,True)
fold_cross_no_shuffle = fold_cross_validation(10, False)
print(fold_cross_shuffle)
print(fold_cross_no_shuffle)


[[0.92352848 0.89692739]
 [0.92849531 0.9028133 ]
 [0.93025848 0.90387984]
 [0.92684271 0.90368819]
 [0.92550282 0.90260123]
 [0.92493113 0.90125026]
 [0.92416321 0.89980057]
 [0.92284285 0.89942751]
 [0.9259066  0.90148709]
 [0.92491415 0.90041515]]
[[0.92352848 0.89692739]
 [0.92849531 0.9028133 ]
 [0.93025848 0.90387984]
 [0.92684271 0.90368819]
 [0.92550282 0.90260123]
 [0.92493113 0.90125026]
 [0.92416321 0.89980057]
 [0.92284285 0.89942751]
 [0.9259066  0.90148709]
 [0.92491415 0.90041515]]


In [16]:
# to avoid running models again:
results_shuffle = [[0.92352848, 0.89692739],
 [0.92849531, 0.9028133 ],
 [0.93025848, 0.90387984],
 [0.92684271, 0.90368819],
 [0.92550282, 0.90260123],
 [0.92493113, 0.90125026],
 [0.92416321, 0.89980057],
 [0.92284285, 0.89942751],
 [0.9259066,  0.90148709],
 [0.92491415, 0.90041515]]

results_no_shuffle = [[0.92831748, 0.89632033],
 [0.92371115, 0.88860198],
 [0.92283015, 0.88648649],
 [0.92514651, 0.88949565],
 [0.92286319, 0.89282879],
 [0.87818062, 0.83895847],
 [0.88789143, 0.85834453],
 [0.89131075, 0.85066274],
 [0.8911651,  0.85913706],
 [0.87837535, 0.82564141]]